In [2]:
%pip install pandas nltk gensim pyLDAvis

import pandas as pd

import sklearn
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim.models import CoherenceModel
from nltk.stem import WordNetLemmatizer, SnowballStemmer
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package wordnet to /Users/Licas/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/Licas/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# chargement du dataset fetch_20newsgroups (4 catégories uniquement)

In [3]:
from sklearn.datasets import fetch_20newsgroups 


categories = [ 'alt.atheism', 'talk.politics.guns', 'sci.space', 'comp.sys.ibm.pc.hardware']
news = fetch_20newsgroups( subset='train', categories=categories, shuffle=True, random_state=42, remove=('headers', 'footers', 'quotes') ) 
df = pd.DataFrame({ 'texte': news.data, 'categorie_id': news.target, 'categorie': [news.target_names[i] for i in news.target] }) 
print(df.shape) 
df.head()

(2209, 3)


,texte,categorie_id,categorie
0,[...]\n\nI am not an expert. My understanding ...,1,comp.sys.ibm.pc.hardware
1,": Frank Crary posted:\n: : Sure, but the diffe...",3,talk.politics.guns
2,C-3's bird may be flaking out and expecting to...,2,sci.space
3,You have missed something. There is a big dif...,2,sci.space
4,"\nFirst of all, I'm not your buddy! Second, r...",0,alt.atheism


In [4]:
df["texte"].isna().sum()

np.int64(0)

In [5]:
df["categorie"].value_counts()

categorie
sci.space                   593
comp.sys.ibm.pc.hardware    590
talk.politics.guns          546
alt.atheism                 480
Name: count, dtype: int64

# après avoir vérifié s'il y avait des valeurs nulles + checker la répartition, on stocke les textes dans une liste 

In [7]:
liste_textes = df["texte"].dropna().tolist()
print(liste_textes[0])

[...]

I am not an expert. My understanding is the watts output of the power 
supply must exceed the sum of the hard disk watts requirement.

Typically, a 200W power supply is sufficient to power a PC.

Hope this help.

Lau Hon-Wah


# il nous faut désormais nettoyer la liste de mots (stopwords + lemmatization)

In [9]:
stemmer = SnowballStemmer('english')

stopwords_list = {
    # Pronoms et articles
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", 
    'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'herself', 'it', "it's", 
    'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'the', 'a', 'an',
    
    # Prépositions et conjonctions
    'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 
    'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 
    'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 
    'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 
    'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 
    'other', 'some', 'such', 'only', 'own', 'same', 'so', 'than', 'too', 'very',
    
    # Verbes auxiliaires et d'état
    'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 
    'having', 'do', 'does', 'did', 'doing', 'can', 'could', 'should', 'would', 'will', 'just'
}


stopwords_eng = STOPWORDS.union(stopwords_list)


def lemmatize_stemming(token):
    lemme= WordNetLemmatizer().lemmatize(token, pos='n')
    return stemmer.stem(lemme)

def preprocess(text):
    tokens = []
    for token in simple_preprocess(text, deacc=True):
        if token not in stopwords_eng and len(token) > 3:
            tokens.append(lemmatize_stemming(token))
    return tokens

# on applique nos fonctions à notre corpus pour les nettoyer

In [10]:
# on applique notre fonction à notre liste de textes pour obtenir "processed_docs"

processed_docs= [preprocess(doc) for doc in liste_textes]

# on récupère les categorie_id de notre df original + tokens
for titre, tokens in zip(df['categorie_id'], processed_docs):
    print(titre, '->', tokens)

1 -> ['expert', 'understand', 'watt', 'output', 'power', 'suppli', 'exceed', 'hard', 'disk', 'watt', 'requir', 'typic', 'power', 'suppli', 'suffici', 'power', 'hope', 'help']
3 -> ['frank', 'crari', 'post', 'sure', 'differ', 'caput', 'crime', 'rate', 'predat', 'control', 'law', 'homicid', 'rate', 'england', 'tenth', 'america', 'england', 'paperwork', 'steve', 'mane', 'ask', 'citat', 'colin', 'greenwood', 'scotland', 'yard', 'studi', 'show', 'control', 'effect', 'crime', 'murder', 'rate', 'book', 'publish', 'london', 'keegan', 'paul', 'misspel', 'disput', 'like', 'richard', 'hofstadt', 'america', 'cultur', 'newton', 'zimr', 'firearm', 'violenc', 'american', 'life', 'statist', 'dissimilar', 'cultur', 'difficult', 'quantifi', 'know', 'state', 'control', 'effect', 'homicid', 'rate', 'accident', 'handgun', 'homicid', 'america', 'licens', 'weapon', 'american', 'child', 'accident', 'shot', 'child', 'year', 'handgun', 'homicid', 'great', 'britain', 'sourc', 'nation', 'safeti', 'council', 'dict

# on construit le dictionnaire Gensin pour obtenir une valeur numérique pour chaque mot



In [11]:
dictionary= gensim.corpora.Dictionary(processed_docs)

print(dictionary.token2id)
print('Nombre de mots gardés :', len(dictionary))

{'disk': 0, 'exceed': 1, 'expert': 2, 'hard': 3, 'help': 4, 'hope': 5, 'output': 6, 'power': 7, 'requir': 8, 'suffici': 9, 'suppli': 10, 'typic': 11, 'understand': 12, 'watt': 13, 'accident': 14, 'amanda': 15, 'america': 16, 'american': 17, 'argument': 18, 'ask': 19, 'belief': 20, 'book': 21, 'brighton': 22, 'brit': 23, 'britain': 24, 'caput': 25, 'child': 26, 'citat': 27, 'clown': 28, 'colin': 29, 'comment': 30, 'comparison': 31, 'control': 32, 'coordin': 33, 'council': 34, 'crari': 35, 'crime': 36, 'crystal': 37, 'cultur': 38, 'dictionari': 39, 'differ': 40, 'difficult': 41, 'disput': 42, 'dissimilar': 43, 'effect': 44, 'emphasi': 45, 'england': 46, 'firearm': 47, 'frank': 48, 'friend': 49, 'glass': 50, 'gotten': 51, 'great': 52, 'greenwood': 53, 'guest': 54, 'handgun': 55, 'hofstadt': 56, 'homicid': 57, 'introduc': 58, 'keegan': 59, 'know': 60, 'law': 61, 'licens': 62, 'life': 63, 'like': 64, 'london': 65, 'mane': 66, 'misspel': 67, 'mistak': 68, 'motif': 69, 'move': 70, 'murder': 7

# on filtre les valeurs extrèmes

In [12]:
# n filtre avec no_below=5, no_above=0.5 et keep_n=3000

dictionary.filter_extremes(
no_below=5, # garder les mots présents dans au moins 1 document
no_above=0.5, # supprimer les mots présents dans plus de 80% des documents
keep_n=3000 # garder au maximum 1000 mots
)
print('Vocabulaire final :', len(dictionary))

Vocabulaire final : 3000


# On convertit en Bag Of Word pour avoir l'id + son nombre d'occurences

In [14]:
bow_corpus= [dictionary.doc2bow(doc) for doc in processed_docs]
for i, bow in enumerate(bow_corpus[:3]):
    print(df.loc[i, 'categorie_id'], '->', bow)

1 -> [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 3), (8, 1), (9, 1), (10, 2), (11, 1), (12, 1), (13, 2)]
3 -> [(14, 2), (15, 3), (16, 2), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 2), (25, 1), (26, 1), (27, 1), (28, 3), (29, 1), (30, 1), (31, 2), (32, 1), (33, 2), (34, 1), (35, 1), (36, 1), (37, 1), (38, 2), (39, 1), (40, 2), (41, 1), (42, 1), (43, 1), (44, 1), (45, 1), (46, 1), (47, 1), (48, 2), (49, 4), (50, 1), (51, 1), (52, 1), (53, 1), (54, 1), (55, 2), (56, 1), (57, 1), (58, 1), (59, 1), (60, 1), (61, 1), (62, 1), (63, 1), (64, 1), (65, 1), (66, 1), (67, 1), (68, 1), (69, 1), (70, 1), (71, 5), (72, 1), (73, 1), (74, 1), (75, 1), (76, 1), (77, 1), (78, 1), (79, 1), (80, 1), (81, 1), (82, 1), (83, 1), (84, 1), (85, 1), (86, 1), (87, 1), (88, 1)]
2 -> [(89, 1), (90, 1), (91, 1), (92, 1), (93, 1), (94, 1), (95, 2), (96, 1), (97, 1), (98, 1), (99, 2)]


# on entraîne le modèle LDA

In [15]:

# on crée le LDA model avec num_topics=4, id2word=dictionary, passes=10 à 20, iterations=100 et random_state=42

lda_model= gensim.models.LdaModel(
corpus=bow_corpus,
id2word=dictionary,
num_topics=4,
random_state=42,
passes= 15,
iterations=100,
alpha='auto',
eta='auto'
)
lda_model.save("modele_lda.gensim")
dictionary.save("dictionnaire_lda.gensim")

In [16]:
for idx, topic in lda_model.print_topics(num_words=10):
    print(f'Topic{idx} -> {topic}')

Topic0 -> 0.019*"drive" + 0.014*"control" + 0.013*"work" + 0.012*"scsi" + 0.011*"card" + 0.010*"know" + 0.009*"problem" + 0.008*"time" + 0.007*"like" + 0.007*"thank"
Topic1 -> 0.015*"peopl" + 0.011*"jesus" + 0.008*"like" + 0.007*"think" + 0.006*"matthew" + 0.006*"know" + 0.006*"time" + 0.006*"thing" + 0.006*"right" + 0.006*"said"
Topic2 -> 0.029*"space" + 0.012*"orbit" + 0.012*"nasa" + 0.011*"launch" + 0.008*"shuttl" + 0.007*"post" + 0.007*"satellit" + 0.006*"mission" + 0.006*"list" + 0.005*"earth"
Topic3 -> 0.008*"year" + 0.008*"like" + 0.007*"time" + 0.005*"kill" + 0.005*"state" + 0.004*"differ" + 0.004*"post" + 0.004*"requir" + 0.004*"thing" + 0.004*"know"


In [17]:
# on renomme les topics à partir des mots présents et de leur fréquence

topic_names= {
0: "IT",
1: "Religion",
2: "Space", 
3: "War"
}

for i, bow in enumerate(bow_corpus):
    distrib= lda_model.get_document_topics(bow)
    print(df.loc[i, 'categorie_id'])
    for topic_id, score in distrib:
        print(" -", topic_names[topic_id], ":", round(score, 3))

1
 - IT : 0.743
 - Space : 0.247
3
 - Religion : 0.298
 - War : 0.7
2
 - Space : 0.59
 - War : 0.395
2
 - IT : 0.367
 - Space : 0.308
 - War : 0.315
0
 - IT : 0.116
 - Religion : 0.882
3
 - IT : 0.197
 - Religion : 0.536
 - War : 0.264
0
 - Religion : 0.987
1
 - IT : 0.719
 - Space : 0.278
1
 - IT : 0.817
 - Religion : 0.043
 - Space : 0.139
3
 - Religion : 0.783
 - Space : 0.214
0
 - IT : 0.082
 - Religion : 0.075
 - Space : 0.052
 - War : 0.791
1
 - IT : 0.621
 - Religion : 0.193
 - Space : 0.185
1
 - IT : 0.67
 - War : 0.321
3
 - Religion : 0.717
 - War : 0.28
0
 - Religion : 0.962
 - Space : 0.035
0
 - Religion : 0.886
 - War : 0.112
2
 - Religion : 0.663
 - Space : 0.335
3
 - IT : 0.214
 - Religion : 0.774
2
 - IT : 0.22
 - Space : 0.58
 - War : 0.197
3
 - IT : 0.17
 - Religion : 0.577
 - War : 0.252
3
 - IT : 0.23
 - Religion : 0.755
0
 - Religion : 0.867
 - War : 0.129
3
 - IT : 0.222
 - Religion : 0.479
 - Space : 0.158
 - War : 0.141
0
 - Religion : 0.625
 - War : 0.363
0
 - I

In [18]:
for i, bow in enumerate(bow_corpus):
    distrib= lda_model.get_document_topics(bow)
    print(df.loc[i, 'categorie_id'], '->', distrib)

1 -> [(0, np.float32(0.7433692)), (2, np.float32(0.24743101))]
3 -> [(1, np.float32(0.29818866)), (3, np.float32(0.69991904))]
2 -> [(2, np.float32(0.5896083)), (3, np.float32(0.3945105))]
2 -> [(0, np.float32(0.3666455)), (2, np.float32(0.30845448)), (3, np.float32(0.31506133))]
0 -> [(0, np.float32(0.11578024)), (1, np.float32(0.88169676))]
3 -> [(0, np.float32(0.19736755)), (1, np.float32(0.5359855)), (3, np.float32(0.26398453))]
0 -> [(1, np.float32(0.98652583))]
1 -> [(0, np.float32(0.7194829)), (2, np.float32(0.27776587))]
1 -> [(0, np.float32(0.81700987)), (1, np.float32(0.042769752)), (2, np.float32(0.13942641))]
3 -> [(1, np.float32(0.78328824)), (2, np.float32(0.21350998))]
0 -> [(0, np.float32(0.08164029)), (1, np.float32(0.07545486)), (2, np.float32(0.05200069)), (3, np.float32(0.7909041))]
1 -> [(0, np.float32(0.62142634)), (1, np.float32(0.19278622)), (2, np.float32(0.18459839))]
1 -> [(0, np.float32(0.6695822)), (3, np.float32(0.32104564))]
3 -> [(1, np.float32(0.7165968

# visualisation avec pyLDAvis

In [19]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

vis = gensimvis.prepare(lda_model, bow_corpus, dictionary)
pyLDAvis.display(vis)